# Transcrição GPU — sop-extractor

Script verificado seguindo as regras de `docs/COLAB_SCRIPT_INCIDENTS.md`.

**IMPORTANTE:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# 1. Instalar dependências
!pip install -q faster-whisper yt-dlp

In [ ]:
# 2. Imports (REGRAS: todos na primeira célula)
import time
import json
import os
import subprocess
from pathlib import Path
from faster_whisper import WhisperModel, BatchedInferencePipeline

print('Imports OK!')

In [ ]:
# 3. Carregar modelo (REGRAS: BatchedInferencePipeline para batch_size)
model = WhisperModel('base', device='cuda', compute_type='int8')
batched_model = BatchedInferencePipeline(model=model)
print('Modelo carregado na GPU!')

In [ ]:
# 4. Vídeos para processar (REGRAS: f-string SEM .format())
VIDEO_IDS = [
    'mYDSSRS-B5U',
]

print(f'{len(VIDEO_IDS)} vídeo(s) para transcrever')

In [ ]:
# 5. Função de transcrição
# REGRAS: subprocess.run() com capture_output=True
# REGRAS: indentação consistente em try/except
def transcribe(video_id):
    url = f'https://www.youtube.com/watch?v={video_id}'
    out = Path(f'output/{video_id}')
    out.mkdir(parents=True, exist_ok=True)
    
    # Download (REGRAS: subprocess.run, NÃO os.system)
    print(f'\nDownload: {video_id}')
    t0 = time.time()
    result = subprocess.run(
        ['yt-dlp', '-x', '--audio-format', 'mp3', '-o', 'temp_audio.%(ext)s', url],
        capture_output=True, text=True, timeout=300
    )
    
    # REGRAS: mostrar erros detalhados
    if result.returncode != 0:
        print(f'  Erro no download:')
        print(f'  {result.stderr[:500]}')
        return None
    
    audio = next(Path('.').glob('temp_audio.*'), None)
    if not audio:
        print('  Arquivo de áudio não encontrado')
        return None
    
    # REGRAS: verificar tamanho do áudio
    print(f'  Áudio: {audio.name} ({audio.stat().st_size / 1024 / 1024:.1f} MB)')
    
    # Transcrever (REGRAS: batched_model.transcribe)
    print(f'Transcrevendo com GPU...')
    t1 = time.time()
    segments, info = batched_model.transcribe(str(audio), batch_size=16, beam_size=5)
    segs = list(segments)
    elapsed = time.time() - t1
    print(f'  {len(segs)} segmentos em {elapsed:.1f}s')
    
    # Salvar SRT (REGRAS: loop explícito, NÃO generator)
    def ts(s):
        h, m, sec = int(s//3600), int((s%3600)//60), int(s%60)
        ms = round((s-int(s))*1000)
        return f'{h:02d}:{m:02d}:{sec:02d},{ms:03d}'
    
    srt_lines = []
    for i, seg in enumerate(segs, 1):
        srt_lines.append(str(i))
        srt_lines.append(f'{ts(seg.start)} --> {ts(seg.end)}')
        srt_lines.append(seg.text.strip())
        srt_lines.append('')
    srt = '\n'.join(srt_lines)
    (out / 'transcript.srt').write_text(srt, encoding='utf-8')
    
    text = '\n'.join(s.text.strip() for s in segs)
    (out / 'full_text.txt').write_text(text, encoding='utf-8')
    
    # Metadata
    meta = {
        'video_id': video_id,
        'language': info.language,
        'duration': info.duration,
        'word_count': len(text.split()),
        'segments': len(segs),
        'gpu_time': elapsed,
    }
    (out / 'metadata.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    
    # Limpar
    audio.unlink(missing_ok=True)
    speedup = info.duration / elapsed if elapsed > 0 else 0
    print(f'  {len(text.split())} palavras em {elapsed:.1f}s ({speedup:.0f}x)')
    return meta

print('Função definida!')

In [ ]:
# 6. Processar todos os vídeos
results = []
t_start = time.time()

for i, vid in enumerate(VIDEO_IDS, 1):
    print(f'\n{"="*50}')
    print(f'VÍDEO {i}/{len(VIDEO_IDS)}')
    print(f'{"="*50}')
    r = transcribe(vid)
    if r:
        results.append(r)

total = time.time() - t_start

In [ ]:
# 7. Relatório
print(f'\n{"="*60}')
print('RELATÓRIO DE TRANSCRIÇÃO')
print(f'{"="*60}')
print(f'\nVídeos: {len(results)}/{len(VIDEO_IDS)}')
print(f'Tempo total GPU: {total:.1f}s')

if results:
    print(f'\n{"Vídeo":<15} {"Duração":>10} {"Palavras":>10}')
    print('-'*35)
    for r in results:
        d = r['duration']/60
        print(f'{r["video_id"]:<15} {d:>9.1f}m {r["word_count"]:>10}')
    print('-'*35)
    total_dur = sum(r['duration'] for r in results)/60
    total_words = sum(r['word_count'] for r in results)
    print(f'{"TOTAL":<15} {total_dur:>9.1f}m {total_words:>10}')
else:
    print('\nNenhum vídeo processado com sucesso.')

In [ ]:
# 8. Download dos resultados
# REGRAS: verificar existência ANTES de criar ZIP
import shutil
from google.colab import files

output_dir = Path('output')
if not output_dir.exists() or not any(output_dir.iterdir()):
    print('Nenhum resultado encontrado. Execute as células anteriores!')
else:
    # Listar arquivos
    print('Arquivos gerados:')
    for f in output_dir.rglob('*'):
        if f.is_file():
            print(f'  {f}')
    
    # Criar ZIP
    shutil.make_archive('transcriptions', 'zip', 'output')
    zip_path = Path('transcriptions.zip')
    print(f'\nZIP criado: {zip_path.stat().st_size / 1024:.1f} KB')
    
    # Baixar
    files.download('transcriptions.zip')